In [0]:
import pyspark.sql.functions as F


In [0]:
input_path = 's3://ims-dbr-raw/green_trip_records/'
output_path = 's3://ims-dbr-stage/green_trip_records/'

In [0]:
df = spark.read.parquet(input_path)
display(df)

#Schema

In [0]:
df_stage = df \
	.withColumn('VendorID', F.col('VendorID').cast('int')) \
	.withColumn('lpep_pickup_datetime', F.col('lpep_pickup_datetime').cast('timestamp')) \
	.withColumn('lpep_dropoff_datetime', F.col('lpep_dropoff_datetime').cast('timestamp')) \
	.withColumn('store_and_fwd_flag', F.col('store_and_fwd_flag').cast('string')) \
	.withColumn('RatecodeID', F.col('RatecodeID').cast('int')) \
	.withColumn('PULocationID', F.col('PULocationID').cast('int')) \
	.withColumn('DOLocationID', F.col('DOLocationID').cast('int')) \
	.withColumn('passenger_count', F.col('passenger_count').cast('int')) \
	.withColumn('trip_distance', F.col('trip_distance').cast('double')) \
	.withColumn('fare_amount', F.col('fare_amount').cast('double')) \
	.withColumn('extra', F.col('extra').cast('double')) \
	.withColumn('mta_tax', F.col('mta_tax').cast('double')) \
	.withColumn('tip_amount', F.col('tip_amount').cast('double')) \
	.withColumn('tolls_amount', F.col('tolls_amount').cast('double')) \
	.withColumn('ehail_fee', F.col('ehail_fee').cast('double')) \
	.withColumn('improvement_surcharge', F.col('improvement_surcharge').cast('double')) \
	.withColumn('total_amount', F.col('total_amount').cast('double')) \
	.withColumn('payment_type', F.col('payment_type').cast('int')) \
	.withColumn('trip_type', F.col('trip_type').cast('int')) \
	.withColumn('congestion_surcharge', F.col('congestion_surcharge').cast('double')) \
    .withColumn('trip_duration', 
                F.timestamp_diff('minute', F.col('lpep_pickup_datetime'), F.col('lpep_dropoff_datetime'))) \
    .withColumn('pu_hour_of_day', F.hour('lpep_pickup_datetime')) \
    .withColumn('pu_day_of_week', F.dayofweek('lpep_pickup_datetime')) \
    .withColumn('pu_year', F.year('lpep_pickup_datetime')) \
    .withColumn('pu_month', F.month('lpep_pickup_datetime')) \
    .withColumn('do_hour_of_day', F.hour('lpep_dropoff_datetime')) \
    .withColumn('do_day_of_week', F.dayofweek('lpep_dropoff_datetime')) \
    .withColumn('do_year', F.year('lpep_dropoff_datetime')) \
    .withColumn('do_month', F.month('lpep_dropoff_datetime')) \
    .withColumn('is_pu_unknown', (F.col('PULocationID') == 264) | (F.col('PULocationID') == 265)) \
    .withColumn('is_do_unknown', (F.col('DOLocationID') == 264) | (F.col('DOLocationID') == 265)) \
    .withColumn('is_pu_unknown', (F.col('PULocationID') == 264) | (F.col('PULocationID') == 265)) \
    .withColumn('is_do_unknown', (F.col('DOLocationID') == 264) | (F.col('DOLocationID') == 265)) \
    .fillna({
    'store_and_fwd_flag' : 'Unknown',
    'RatecodeID' : 1,
    'passenger_count' : 1,
    'ehail_fee' : 0,
    'payment_type' : 1,
    'trip_type' : 1,
    'congestion_surcharge' : 0}) \
    .filter(
    (F.col('fare_amount') >0 ) & 
    (F.col('passenger_count')>0) & 
    (F.col('trip_distance')>0) & 
    (F.col('trip_duration')>0)) \
    .dropDuplicates(subset = ['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID'])





#Write to Silver Table

In [0]:
## Write to Stage Table
df_stage.write \
  .mode("overwrite") \
  .parquet(output_path)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS nytaxi.staging.green_trip_records
USING PARQUET
LOCATION 's3://ims-dbr-stage/green_trip_records/';